# Fraudulent Job Advertisement Detection  
### Multi-Dataset Machine Learning Project

## Project Overview

Online job  and messaging platforms are increasingly targeted by fraudulent bad actors with advertisements designed to deceive applicants, extract personal information, or collect financial payments. Detecting such scams is a relevant real-world machine learning problem involving natural language processing and classification.

This project aims to develop a machine learning model capable of distinguishing between legitimate and fraudulent job advertisements using textual data.

To ensure consistency and minimise dataset-specific bias, three separate job advertisement datasets are cleaned, aligned to a shared format, and combined into a single modelling framework.

Each dataset will be transformed into a shared structure consisting of:

- `id` – unique identifier  
- `text` – consolidated textual representation of the advert  
- `label` – binary target (1 = fraudulent, 0 = legitimate)  
- `source` – dataset origin  

This notebook outlines the full machine learning pipeline for detecting fraudulent job advertisements, including data standardisation across multiple datasets, feature construction, model development, and evaluation


---

### Student Information

| Name            | Student ID   |
|-----------------|-------------|
| Alice Brennan   | X00228770   |
| Cillian Murphy  | X00226829   |
| Wael Eladham    | X00227390   |

---

Alice

In [ ]:
import pandas as pd

zip_path = "Job Adverts Dataset.zip"
df_rawdata = pd.read_csv(zip_path, compression="zip")
df_rawdata.shape

(17880, 18)

In [4]:
df_rawdata = pd.read_csv("Job Adverts Dataset.zip", compression="zip")

In [ ]:
df = df_rawdata.copy()

# Normalise column names
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
def safe(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.strip()
# Unified text feature
df["text"] = (
    "TITLE: " + safe(df["title"]) + " "
    "COMPANY_PROFILE: " + safe(df["company_profile"]) + " "
    "DESCRIPTION: " + safe(df["description"]) + " "
    "REQUIREMENTS: " + safe(df["requirements"]) + " "
    "BENEFITS: " + safe(df["benefits"])
).str.replace(r"\s+", " ", regex=True).str.strip()
# Final standard output
df_clean = pd.DataFrame({
    "id": df["job_id"].astype(str),
    "text": df["text"],
    "label": df["fraudulent"].astype(int),
    "source": "emscad"
})
df_clean.head()

,id,text,label,source
0,1,TITLE: Marketing Intern COMPANY_PROFILE: We're...,0,emscad
1,2,TITLE: Customer Service - Cloud Video Producti...,0,emscad
2,3,TITLE: Commissioning Machinery Assistant (CMA)...,0,emscad
3,4,TITLE: Account Executive - Washington DC COMPA...,0,emscad
4,5,TITLE: Bill Review Manager COMPANY_PROFILE: Sp...,0,emscad


In [6]:
print("Shape:", df_clean.shape)
print("\nLabel counts:\n", df_clean["label"].value_counts())
print("\nNulls:\n", df_clean.isnull().sum())
print("\nText length summary:\n", df_clean["text"].str.len().describe())

df_clean.to_csv("alice_clean.csv", index=False)
print("\nSaved: alice_clean.csv")

Shape: (17880, 4)

Label counts:
 label
0    17014
1      866
Name: count, dtype: int64

Nulls:
 id        0
text      0
label     0
source    0
dtype: int64

Text length summary:
 count    17880.000000
mean      2713.972539
std       1447.209496
min         75.000000
25%       1648.000000
50%       2568.000000
75%       3514.000000
max      14942.000000
Name: text, dtype: float64

Saved: alice_clean.csv


### Cillian

In [7]:
jobData_raw = pd.read_csv('jobData.csv', sep = ',')  

In [8]:
# Find rows with NaNs
nanData = jobData_raw.isna().any()
print("NaN:\n", nanData, "\n", sep='')

# Use mask to remove NaN rows
mask = (jobData_raw.isna()).any(axis=0)
jobData = jobData_raw.loc[:, ~mask]

jobData["text"] = (
    "TITLE: " + jobData["job_title"] + " "
    "COMPANY_PROFILE: " + jobData["company_profile"] + " "
    "DESCRIPTION: " + jobData["job_description"] + " "
    "REQUIREMENTS: " + jobData["requirements"]
).str.replace(r"\s+", " ", regex=True).str.strip()


final_df_cillian = pd.DataFrame({
    "id": jobData["job_id"].astype(str),
    "text": jobData["text"],
    "label": jobData["is_fake"].astype(int),
    "source": ""
})

final_df_cillian.head()

NaN:
job_id                       False
job_title                    False
job_description              False
requirements                 False
benefits                     False
company_name                  True
company_profile              False
industry                     False
employment_type              False
location                     False
salary_range                 False
required_experience_years    False
education_level              False
department                   False
posting_date                 False
application_deadline         False
contact_email                False
company_website               True
has_logo                     False
num_open_positions           False
job_function                 False
telecommuting                False
fraud_reason                  True
text_length                  False
is_fake                      False
dtype: bool



C:\Users\alice\AppData\Local\Temp\ipykernel_11392\1433624960.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jobData["text"] = (


,id,text,label,source
0,1,TITLE: Software Engineer COMPANY_PROFILE: Our ...,0,
1,2,TITLE: Content Writer COMPANY_PROFILE: Our com...,0,
2,3,TITLE: Customer Support Specialist COMPANY_PRO...,1,
3,4,TITLE: Data Analyst COMPANY_PROFILE: Our compa...,0,
4,5,TITLE: Graphic Designer COMPANY_PROFILE: We ar...,1,


In [9]:
print(final_df_cillian.shape)
nanData = final_df_cillian.isna().any()
print("NaN:\n", nanData, "\n", sep='')

(3000, 4)
NaN:
id        False
text      False
label     False
source    False
dtype: bool



### Wael

## Merging Datasets

In [ ]:
# features = []

# # Get features for merging
# for col in final_df_cillian.columns.values:
#     features.append(col)

# # Change to actuall df names when ready
# mergedData = pd.merge(df_clean, final_df_cillian, WaelDf, alice_clean,on = features, how = "outer") 

# mergedData = pd.merge(df_clean, final_df_cillian, WaelDf, alice_clean, on=features, how="outer")
alice_df = df_clean.copy()
cillian_df = final_df_cillian.copy()

required_cols = ["id", "text", "label", "source"]

def enforce_schema(d):
    d = d.copy()
    for col in required_cols:
        if col not in d.columns:
            d[col] = ""
    return d[required_cols]

alice_df = enforce_schema(alice_df)
cillian_df = enforce_schema(cillian_df)

# Fix sources so they clearly reflect contributors
alice_df["source"] = alice_df["source"].replace({"": "alice_emscad", "emscad": "alice_emscad"})
cillian_df["source"] = cillian_df["source"].replace({"": "cillian_dataset"})

mergedData = pd.concat([alice_df, cillian_df], ignore_index=True)

print("Merged shape:", mergedData.shape)
print("\nColumns:", mergedData.columns.tolist())
print("\nBy source:\n", mergedData["source"].value_counts(dropna=False))
print("\nBy label:\n", mergedData["label"].value_counts(dropna=False))

# checking for duplicate ids
dup_ids = mergedData["id"].duplicated().sum()
print("Duplicate ids:", dup_ids)

# checking for empty text rows
empty_text = (mergedData["text"].astype(str).str.strip() == "").sum()
print("Empty text rows:", empty_text)

# saving merged datasets (Alice, Cillian; add Wael later)
mergedData.to_csv("merged_alice_cillian.csv", index=False)
print("Saved: merged_alice_cillian.csv")


Merged shape: (20880, 4)

Columns: ['id', 'text', 'label', 'source']

By source:
 source
alice_emscad       17880
cillian_dataset     3000
Name: count, dtype: int64

By label:
 label
0    18542
1     2338
Name: count, dtype: int64
Duplicate ids: 3000
Empty text rows: 0
Saved: merged_alice_cillian.csv


Summary of merged datasets (Alice, Cillian, Wael tbc)

In [24]:
print("Fraud percentage:", round(mergedData["label"].mean()*100, 2), "%")
print("\nDataset breakdown:")
print(mergedData.groupby("source")["label"].value_counts())

Fraud percentage: 11.2 %

Dataset breakdown:
source           label
alice_emscad     0        17014
                 1          866
cillian_dataset  0         1528
                 1         1472
Name: count, dtype: int64


In [ ]:
y = mergedData['label'].astype(int)
X = mergedData.loc[ : , ~mergedData.columns.isin(['label'])] # add source to here if you dont want to include it